# Phase 25a verification -- full `eval_calc_v4` re-run after both fixes

Phase 25a fixed two things purely by re-testing the single reported failure case
(word-form Arabic years for `extract_service`; the gender completeness guard).
This notebook is the follow-up that was still owed: a **fresh, full 8-question x 4-trial
re-run of `eval/eval_calc_v4.jsonl`** (Track 2 of `scripts/25. revaluation_v4.ipynb`)
against the Phase 25a code, so the fix is confirmed under the same harness Phase 25
used rather than one hand-picked question.

Writes to **new** files (`phase25a_calc_*.csv`) -- Phase 25's original
`phase25_calc_*.csv` files are left untouched as the historical pre-fix record.
No LLM judge is used here (same as Phase 25 Track 2): grading is a structural check
against `res["calculation"]` (the deterministic block) plus a keyword/number check
against the narrated answer text, since the block being correct does not guarantee
the model states it correctly (see verdict below).

In [1]:
import json
import re as _re
import sys
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
sys.path.insert(0, str(PROJECT))

from src.retrieval import Retriever, FAMILY_TRIGGER_DEPTH, FAMILY_TRIGGER_COUNT
from src.generation import get_production_generator, SYSTEM_PROMPT

EVAL_CALC = PROJECT / "eval" / "eval_calc_v4.jsonl"
CALC_RETR_PATH = PROJECT / "eval" / "phase25a_calc_retrieval.csv"
CALC_GEN_PATH = PROJECT / "eval" / "phase25a_calc_generations.csv"
GRADING_PATH = PROJECT / "eval" / "phase25a_calc_grading.csv"

N_TRIALS = 4

evc = pd.DataFrame([json.loads(l) for l in open(EVAL_CALC, encoding="utf-8")])

n_rules = len(_re.findall(r"^\d+\.", SYSTEM_PROMPT, _re.MULTILINE))
assert n_rules == 20, f"expected the 20-rule Phase 24e prompt, found {n_rules}"
assert FAMILY_TRIGGER_DEPTH == 37, f"expected the Phase 25 retrieval fix (depth 37), found {FAMILY_TRIGGER_DEPTH}"

print(f"eval_calc_v4: {len(evc)} questions | prompt rules: {n_rules} | "
      f"family_trigger_depth={FAMILY_TRIGGER_DEPTH} count={FAMILY_TRIGGER_COUNT}")

eval_calc_v4: 8 questions | prompt rules: 20 | family_trigger_depth=37 count=2


## Retrieval (fresh, once per question, cached)

In [2]:
if CALC_RETR_PATH.exists():
    cretr = pd.read_csv(CALC_RETR_PATH, encoding="utf-8-sig")
    cretr["chunk_ids"] = cretr["chunk_ids"].apply(json.loads)
    print(f"loaded cached retrieval for {len(cretr)} questions")
else:
    calc_retriever = Retriever()
    rows = []
    for q in evc.itertuples(index=False):
        chunks = calc_retriever.retrieve(q.question_text)
        want = {str(a) for a in q.answer_article_numbers}
        found = {c["article_number"] for c in chunks if c["law"] == q.answer_law
                 and c["language"] == q.answer_language and c["article_number"] in want}
        rows.append({"question_id": q.question_id, "chunk_ids": [c["chunk_id"] for c in chunks],
                     "full_coverage": found == want, "n_found": len(found)})
    cretr = pd.DataFrame(rows)
    out = cretr.copy()
    out["chunk_ids"] = out["chunk_ids"].apply(json.dumps)
    out.to_csv(CALC_RETR_PATH, index=False, encoding="utf-8-sig")

print(f"retrieval full coverage: {cretr.full_coverage.sum()}/{len(cretr)}")

chunks_tbl = pd.read_parquet(PROJECT / "0. cleaned_data" / "chunks_articles.parquet")
chunks_tbl["article_number"] = chunks_tbl["article_number"].astype(str)
BY_ID = {c.chunk_id: c for c in chunks_tbl.itertuples()}


def calc_chunks_for(qid):
    ids = cretr.loc[cretr.question_id == qid, "chunk_ids"].iloc[0]
    return [{"chunk_id": i, "law": BY_ID[i].law, "language": BY_ID[i].language,
             "article_number": BY_ID[i].article_number, "text": BY_ID[i].text} for i in ids]

loaded cached retrieval for 8 questions
retrieval full coverage: 8/8


## Generation -- 4 trials per question, production temperature

Same reasoning as Phase 25: whether the model *narrates* a correct computed block
correctly is a separate question from the block being correct (Phase 24e: the
gender block was present 6/6 but only stated both figures 2/6 before a prompt fix).
Resumable -- re-running this notebook after the first pass costs nothing.

In [3]:
calc_rows = []
if CALC_GEN_PATH.exists():
    calc_rows = pd.read_csv(CALC_GEN_PATH, encoding="utf-8-sig").to_dict("records")
done_trials = {(r["question_id"], r["trial"]) for r in calc_rows}
todo = [(q.question_id, t) for q in evc.itertuples(index=False) for t in range(1, N_TRIALS + 1)
        if (q.question_id, t) not in done_trials]
print(f"{len(done_trials)} trials already saved, {len(todo)} to go")

if todo:
    calc_generator = get_production_generator()
    for qid, trial in todo:
        q = evc[evc.question_id == qid].iloc[0]
        ch = calc_chunks_for(qid)
        res = calc_generator.generate(q.question_text, ch)
        calc_rows.append({
            "question_id": qid, "trial": trial, "question_text": q.question_text,
            "question_language": q.question_language, "question_category": q.question_category,
            "answer": res["answer"],
            "calculation": json.dumps(res.get("calculation"), ensure_ascii=False),
        })
        pd.DataFrame(calc_rows).to_csv(CALC_GEN_PATH, index=False, encoding="utf-8-sig")
        print(f"  generated {len(calc_rows)}/{len(evc) * N_TRIALS}")

calc_gen = pd.DataFrame(calc_rows)
print(f"\n{len(calc_gen)} trials ready")

32 trials already saved, 0 to go

32 trials ready


## Grading -- identical logic to `scripts/25. revaluation_v4.ipynb` cell 16

Graded against `res["calculation"]` directly (the deterministic structure
`compute_contributions`/`compute_end_of_service`/`compute_gender_conditions`
actually returned), plus a check that the number/behaviour it requires is present
in the *narrated* answer text.

In [4]:
def get_calc(row):
    raw = row["calculation"]
    if pd.isna(raw) or raw in (None, "null"):
        return None
    return json.loads(raw)


def get_contrib(calc):
    if calc is None:
        return None
    if "employer_rate" in calc:
        return calc
    return calc.get("contributions")


def get_eos(calc):
    return calc.get("end_of_service") if calc else None


def get_gender(calc):
    return calc.get("gender_conditions") if calc else None


AR_DIGITS = str.maketrans("\u0660\u0661\u0662\u0663\u0664\u0665\u0666\u0667\u0668\u0669", "0123456789")


def has_number(text, value, tol=0.01):
    t = str(text).translate(AR_DIGITS)
    for cand in (f"{value:g}", f"{value:.1f}", f"{value:.2f}", f"{int(value)}" if value == int(value) else None):
        if cand and cand in t:
            return True
    return False


def check_c01(calc, ans):
    c = get_contrib(calc)
    calc_ok = bool(c and abs(c["employer_rate"] - 14.25) < 0.01 and c.get("employer_amount") == 142.5)
    ans_ok = has_number(ans, 142.5) or has_number(ans, 14.25)
    return calc_ok, ans_ok


def check_c02(calc, ans):
    c = get_contrib(calc)
    calc_ok = bool(c and abs(c["employee_rate"] - 7.5) < 0.01 and c.get("employee_amount") == 75.0)
    ans_ok = has_number(ans, 75.0) or has_number(ans, 7.5)
    return calc_ok, ans_ok


def check_c03(calc, ans):
    c = get_contrib(calc)
    calc_ok = bool(c and c.get("capped") and c.get("ceiling_exception_applies")
                   and c.get("ceiling") == 3000.0 and c.get("ceiling_exception") == 5000.0)
    ans_ok = ("15" in ans and "10" in ans and "2009" in ans) or ("5000" in ans.replace(",", "")) or ("5,000" in ans)
    return calc_ok, ans_ok


def check_c04(calc, ans):
    e = get_eos(calc)
    calc_ok = bool(e and e.get("months_owed") == 10 and e.get("amount") == 8000.0 and e.get("requires_not_ssl"))
    asks = any(kw in ans for kw in ["\u0627\u0644\u0636\u0645\u0627\u0646 \u0627\u0644\u0627\u062c\u062a\u0645\u0627\u0639\u064a", "Social Security", "\u062e\u0627\u0636\u0639", "subject"])
    ans_ok = has_number(ans, 8000.0) and asks
    return calc_ok, ans_ok


def check_gender(calc, ans, expect_sex=None):
    g = get_gender(calc)
    have_62 = any(x["article"] == "62" for x in (g or []))
    have_64 = any(x["article"] == "64" for x in (g or []))
    calc_ok = bool(g and have_62 and have_64
                   and next(x for x in g if x["article"] == "62")["male"] == 60
                   and next(x for x in g if x["article"] == "62")["female"] == 55
                   and next(x for x in g if x["article"] == "64")["male"] == 252
                   and next(x for x in g if x["article"] == "64")["female"] == 228)
    both = (has_number(ans, 60) or has_number(ans, 52)) and (has_number(ans, 55) or has_number(ans, 28))
    asked = ("\u061f" in ans and ("\u0630\u0643\u0631" in ans or "male" in ans.lower())) or ("?" in ans and "male" in ans.lower())
    silently_one = not both and not asked
    return calc_ok, (both or asked) and not silently_one


def check_c08(calc, ans):
    c = get_contrib(calc)
    names = [comp.get("name") for comp in (c or {}).get("components", [])] if c else []
    calc_ok = bool(c and all(names) and len(names) == 6)
    markers = ["\u0625\u0635\u0627\u0628", "\u0623\u0645\u0648\u0645\u0629", "\u062a\u0639\u0637\u0644", "\u0634\u064a\u062e\u0648\u062e\u0629", "Injury", "Maternity", "Unemployment", "Old-Age", "Old Age"]
    ans_ok = sum(1 for m in markers if m in ans) >= 3
    return calc_ok, ans_ok


CHECKS = {
    "c01": check_c01, "c02": check_c02, "c03": check_c03, "c04": check_c04,
    "c05": lambda calc, ans: check_gender(calc, ans), "c06": lambda calc, ans: check_gender(calc, ans),
    "c07": lambda calc, ans: check_gender(calc, ans), "c08": check_c08,
}

results = []
for row in calc_gen.itertuples(index=False):
    calc = get_calc(dict(row._asdict()))
    calc_ok, ans_ok = CHECKS[row.question_id](calc, str(row.answer))
    results.append({"question_id": row.question_id, "trial": row.trial,
                     "calc_correct": calc_ok, "narration_correct": ans_ok})

rdf = pd.DataFrame(results)
rdf.to_csv(GRADING_PATH, index=False, encoding="utf-8-sig")

print("=" * 74)
print(f"PHASE 25a VERIFICATION -- Track 2, calculator, {N_TRIALS} trials per question")
print("=" * 74)
summary = rdf.groupby("question_id").agg(
    calc_correct=("calc_correct", "sum"), narration_correct=("narration_correct", "sum"),
    n=("trial", "count"))
print(summary.to_string())
print(f"\ncalc structure correct: {rdf.calc_correct.sum()}/{len(rdf)}")
print(f"narration correct:      {rdf.narration_correct.sum()}/{len(rdf)}")

PHASE 25a VERIFICATION -- Track 2, calculator, 4 trials per question
             calc_correct  narration_correct  n
question_id                                    
c01                     4                  4  4
c02                     4                  4  4
c03                     4                  4  4
c04                     4                  4  4
c05                     4                  4  4
c06                     4                  3  4
c07                     4                  4  4
c08                     4                  4  4

calc structure correct: 32/32
narration correct:      31/32


## Verdict

**Both Phase 25a fixes hold under a full, independent re-run** -- Phase 25a itself
only re-tested the exact reported failure case for each fix; this is the first time
either fix has been checked against the whole `eval_calc_v4` harness.

- **c04 (end-of-service, word-form Arabic years): 0/4 -> 4/4 calc-correct, 0/4 -> 4/4
  narration-correct.** Every trial now conditions the 8,000 JD figure on SSL
  non-coverage and asks or states the precondition, instead of the pre-Phase-24a
  failure mode (a confident wage x years answer for a majority-inapplicable provision).
- **c01/c02/c03/c05/c07/c08: 4/4 calc-correct and 4/4 narration-correct**, unchanged
  from Phase 25 (c07's narration moved 3/4 -> 4/4, within normal trial-to-trial
  sampling variance, not attributed to either fix).
- **c06: 4/4 calc-correct, 3/4 narration-correct.** The gender-completeness guard
  fix (`compute_gender_conditions` now requires *both* Article 62 and 64) is working
  as intended -- the block itself is correct every trial. The one narration miss is
  **not a code bug**: rule 20 explicitly instructs the model to state both figures
  when the block gives both, and one trial (out of four, at production temperature)
  simply didn't follow that instruction, stating only the male threshold. This
  matches Phase 24e's own disclosed ~1-in-6 residual rate for this exact failure mode.

**Total: calc-structure 32/32 (was 24/32 pre-fix), narration 31/32 (was 26/32 pre-fix).**

**Why the gender narration gap is not "solved":** the two fixes in this notebook
(c04's `extract_service`, c06's completeness guard) were both **code** bugs -- a
deterministic function given an input it didn't handle, fixable by fixing the
function, and now verified at 100%. The residual c06 gap is a **different kind of
problem**: the correct data is in the prompt every time (calc-correct is 4/4), and
the failure is the LLM occasionally not following a stated instruction when writing
the final answer. No code fix can make an LLM's free-text generation 100% reliable
under sampling; the prompt rule (20) narrows the failure rate, it does not eliminate
it (Phase 24e / `LESSONS.md` 5.4). Closing this the rest of the way would need a
**new feature** -- a post-generation validator that checks both figures appear in
the answer text and forces a retry or programmatic append if not -- not attempted
here, since it wasn't asked for and changes the shape of the pipeline (an output
guard, not a prompt or calculator change).